# 04 Metrics (Prompt 09, cycle 4)

No GPU needed (CPU is fine). First upload into the Drive project folder: `src/metrics.py`, `tests/test_metrics.py`, `notebooks/04_metrics.ipynb`. Then Runtime → **Restart session** → **Run all**, and copy all the output back into the chat.

In [ ]:
# 1. Drive and project folder
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/atr_colab/AITR_Lab02_MCL2026002_PromptStability

In [ ]:
# 2. Hand-computed unit tests
!python tests/test_metrics.py

In [ ]:
# 3. Independent cross-check against scikit-learn and SciPy on 1,000 random cases
import random, sys
sys.path.insert(0, '.')
from src import metrics as M
from sklearn.metrics import f1_score, cohen_kappa_score, accuracy_score
from scipy.stats import binomtest
rng = random.Random(0)
worst = {'acc': 0, 'f1': 0, 'kappa': 0, 'mcnemar': 0}
for _ in range(1000):
    n = rng.randint(5, 60)
    g = [rng.choice(M.GOLD_LABELS) for _ in range(n)]
    p = [rng.choice(M.GOLD_LABELS) for _ in range(n)]   # no 'invalid' here, so sklearn's definition is the same
    worst['acc'] = max(worst['acc'], abs(M.accuracy(g, p) - accuracy_score(g, p)))
    worst['f1'] = max(worst['f1'], abs(M.macro_f1(g, p) - f1_score(g, p, labels=list(M.GOLD_LABELS), average='macro', zero_division=0)))
    if len(set(g)) > 1 or len(set(p)) > 1:
        worst['kappa'] = max(worst['kappa'], abs(M.cohen_kappa(g, p) - cohen_kappa_score(g, p)))
    a = [rng.random() < 0.6 for _ in range(n)]; b = [rng.random() < 0.6 for _ in range(n)]
    bb, cc, pm = M.mcnemar_exact(a, b)
    ref = binomtest(bb, bb + cc, 0.5).pvalue if bb + cc else 1.0
    worst['mcnemar'] = max(worst['mcnemar'], abs(pm - ref))
print('largest absolute difference from the reference libraries:', worst)
print('PASS' if max(worst.values()) < 1e-9 else 'FAIL')